## Final Model for Error Analysis

### Which model you are analyzing?
- Currently I am analyzing XGBoost mode which predicts that customer will default their payment next month or NOT.

### Why this model was selected?
- Because this model is giving best roc_auc and recall values among others (Gradient Boosting and Random Forest) that is why i am using this model.

### Which threshold is being used?
- Threshold is 0.5 which is default because we did not changed it yet.

In [35]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

xgb_final_model = joblib.load("xgb_balanced.joblib")

df = df = pd.read_csv("cleaned_data.csv", index_col="id")

x = df.drop(columns=["default_payment_next_month"])
y = df["default_payment_next_month"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [36]:
y_prob = xgb_final_model.predict_proba(x_test)[:, 1]
y_pred = xgb_final_model.predict(x_test)

result = x_test.copy()
result["y_true"] = y_test.values
result["y_pred"] = y_pred
result["y_prob"] = y_prob

result.head()

,limit_bal,sex,education,marriage,age,pay_1,pay_2,pay_3,pay_4,pay_5,pay_6,bill_amt1,bill_amt2,bill_amt3,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,y_true,y_pred,y_prob
id,,,,,,,,,,,,,,,,,,,,,,,,,,
8688,50000,2,3,1,43,0,0,0,0,0,0,39177,39607,17070,13038,8904,4740,2000,1500,3500,600,500,4000,0,0,0.353335
24202,240000,1,2,1,31,0,0,0,0,0,0,168376,172809,176012,179175,180809,184383,7648,7900,8000,6500,6900,7000,0,0,0.221826
27783,90000,2,1,2,25,0,0,0,0,0,0,6090,7477,8848,10135,11731,8138,1500,1500,1500,2000,1500,1000,0,0,0.371585
14948,50000,2,2,2,22,0,0,2,0,0,0,30631,30479,28301,26829,27173,26424,4809,0,1200,1500,943,1142,0,0,0.439358
1487,230000,1,1,2,32,0,0,0,0,0,0,44734,47178,29582,38426,42500,43531,10120,20000,10000,5000,5000,5000,0,0,0.060581


In [37]:
(result["y_pred"] == (result["y_prob"] >= 0.5).astype("int64")).value_counts()

True    5993
Name: count, dtype: int64

## Categorize Errors

In [42]:
result["error_type"] = "correct"

result.loc[(result["y_true"] == 1) & (result["y_pred"] == 0), "error_type"] = "false_negative"

result.loc[(result["y_true"] == 0) & (result["y_pred"] == 1), "error_type"] = "false_positive"

result["error_type"].value_counts()

error_type
correct           4535
false_positive     968
false_negative     490
Name: count, dtype: int64

### Which error dominates?
- false positive is the error that dominates.

### Why this is expected for this problem?
- Because we are keeping recall high which in return is flagging non-defaulters as defaulters. So that is the reason their are more false positive.

### Which error is more costly?
- Of course false negative is more costly because it means that defaulters are being flagged as non-defaulters which is direct loss to business and thus more costly.

## Inspecting Errors

In [48]:
result[result.error_type == "false_negative"].head(3)

,limit_bal,sex,education,marriage,age,pay_1,pay_2,pay_3,pay_4,pay_5,pay_6,bill_amt1,bill_amt2,bill_amt3,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,y_true,y_pred,y_prob,error_type
id,,,,,,,,,,,,,,,,,,,,,,,,,,,
1072,150000,2,2,1,44,-1,-1,-1,-1,-1,-1,390,5104,10318,5775,14441,780,5104,10318,10355,14441,780,66950,1,0,0.257188,false_negative
29081,120000,2,2,1,54,1,-2,-2,-2,-1,-1,-200,-200,-200,0,700,1935,0,0,200,700,1935,0,1,0,0.477090,false_negative
13754,180000,1,1,2,30,1,-2,-2,-2,-1,-1,-203,-698,-193,-688,817,1157,0,1000,0,2000,1000,2000,1,0,0.263349,false_negative


In [51]:
result[result.error_type == "false_positive"].head(3)

,limit_bal,sex,education,marriage,age,pay_1,pay_2,pay_3,pay_4,pay_5,pay_6,bill_amt1,bill_amt2,bill_amt3,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,y_true,y_pred,y_prob,error_type
id,,,,,,,,,,,,,,,,,,,,,,,,,,,
23515,120000,2,2,1,30,2,-1,3,2,0,-1,1248,1701,1551,1410,479,3158,1701,0,9,0,3158,0,0,1,0.881077,false_positive
14933,20000,1,2,1,26,1,2,2,2,2,2,11338,10870,12380,11884,12894,12540,0,2000,0,1203,0,1700,0,1,0.883030,false_positive
17588,70000,2,2,2,38,1,2,2,2,2,2,21047,22102,22949,23275,23596,23078,1700,1500,1000,1000,0,2100,0,1,0.801147,false_positive
